## 1. Environment Setup

In [1]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

import json
import os
import sys

import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("\nLibraries loaded successfully!")

Python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Pandas version: 3.0.5
NumPy version: 2.5.3

Libraries loaded successfully!


In [2]:
# ============================================
# 2. CHECK PYTHON ENVIRONMENT
# ============================================

print("Python executable:")
print(sys.executable)

print("\nCurrent working directory:")
print(os.getcwd())

Python executable:
c:\DATA ANALYSIS PROJECTS\Yelp_Business_Analytics\.venv\Scripts\python.exe

Current working directory:
c:\DATA ANALYSIS PROJECTS\Yelp_Business_Analytics


## 2. Project Data Paths

In [3]:
# ============================================
# 3. CREATE PROJECT FOLDERS
# ============================================

folders = [
    "data",
    "data/raw",
    "data/processed",
    "data/curated",
    "output",
    "sql",
    "reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


## 3. Data Acquisition

The Yelp Business dataset was acquired using the **Kaggle API**.

In [4]:
%pip install -U kagglehub

Note: you may need to restart the kernel to use updated packages.


### Kaggle API Connection

The following optional cell demonstrates the Kaggle API connection used to acquire the Business JSON file. It requires Kaggle API credentials to be configured locally.

In [5]:
# ============================================
# STEP 3 — LOAD YELP BUSINESS DATA SAFELY
# ============================================

import pandas as pd
from pathlib import Path

business_file = Path("data/raw/yelp_academic_dataset_business.json")

# Check file
if not business_file.exists():
    raise FileNotFoundError(
        f"Business file not found: {business_file.resolve()}"
    )

print("Business file found:")
print(business_file.resolve())

# Read JSON Lines in chunks
business_reader = pd.read_json(
    business_file,
    lines=True,
    chunksize=5_000
)

print("\nYelp Business JSON reader created successfully.")
print("Chunk size: 5,000 rows")

Business file found:
C:\DATA ANALYSIS PROJECTS\Yelp_Business_Analytics\data\raw\yelp_academic_dataset_business.json

Yelp Business JSON reader created successfully.
Chunk size: 5,000 rows


## 4. Load and Profile the Business Data

A chunked JSON reader is used for initial profiling before the complete Business dataset is loaded.

In [6]:
# Read first chunk only
business_sample = next(business_reader)

print("Rows:", len(business_sample))
print("Columns:", len(business_sample.columns))

business_sample.head()

Rows: 5000
Columns: 14


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [7]:
# ============================================
# STEP 4.1 — BASIC DATA PROFILING
# ============================================

print("Shape:")
print(business_sample.shape)

print("\nColumn Names:")
print(business_sample.columns.tolist())

print("\nData Types:")
print(business_sample.dtypes)

print("\nMemory Usage:")
print(
    f"{business_sample.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Shape:
(5000, 14)

Column Names:
['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

Data Types:
business_id         str
name                str
address             str
city                str
state               str
postal_code         str
latitude        float64
longitude       float64
stars           float64
review_count      int64
is_open           int64
attributes       object
categories          str
hours            object
dtype: object

Memory Usage:
4.98 MB


In [8]:
# ============================================
# STEP 4.2 — MISSING VALUE ANALYSIS
# ============================================

missing = business_sample.isnull().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": (
        missing / len(business_sample) * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

missing_summary

,missing_count,missing_percentage
hours,782,15.64
attributes,441,8.82
categories,5,0.10
address,0,0.00
business_id,0,0.00
name,0,0.00
postal_code,0,0.00
state,0,0.00
city,0,0.00
latitude,0,0.00


In [9]:
# ============================================
# STEP 4.3 — NUMERICAL DATA PROFILING
# ============================================

business_sample[
    ["latitude", "longitude", "stars", "review_count", "is_open"]
].describe()

,latitude,longitude,stars,review_count,is_open
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,36.768674,-89.495122,3.588300,46.918600,0.790400
std,5.956277,14.978828,0.977826,129.039502,0.407064
min,27.584300,-119.968230,1.000000,5.000000,0.000000
25%,32.206523,-90.358929,3.000000,8.000000,1.000000
50%,38.792358,-86.136475,3.500000,15.000000,1.000000
75%,39.952842,-75.398866,4.500000,40.000000,1.000000
max,53.647812,-74.658572,5.000000,4554.000000,1.000000


In [10]:
# ============================================
# STEP 4.4 — CATEGORICAL DATA PROFILING
# ============================================

print("Unique business IDs:", business_sample["business_id"].nunique())
print("Total rows:", len(business_sample))

print("\nUnique cities:", business_sample["city"].nunique())
print("Unique states:", business_sample["state"].nunique())

print("\nTop 10 states:")
print(business_sample["state"].value_counts().head(10))

print("\nOpen / Closed:")
print(business_sample["is_open"].value_counts())

Unique business IDs: 5000
Total rows: 5000

Unique cities: 454
Unique states: 14

Top 10 states:
state
PA    1128
FL     850
TN     424
IN     380
AZ     348
MO     335
LA     329
NJ     295
NV     276
AB     213
Name: count, dtype: int64

Open / Closed:
is_open
1    3952
0    1048
Name: count, dtype: int64


In [11]:
# ============================================
# STEP 4.5 — CATEGORY PROFILING
# ============================================

print("Sample category values:\n")

for value in business_sample["categories"].dropna().head(10):
    print(value)

Sample category values:

Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists
Shipping Centers, Local Services, Notaries, Mailbox Centers, Printing Services
Department Stores, Shopping, Fashion, Home & Garden, Electronics, Furniture Stores
Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries
Brewpubs, Breweries, Food
Burgers, Fast Food, Sandwiches, Food, Ice Cream & Frozen Yogurt, Restaurants
Sporting Goods, Fashion, Shoe Stores, Shopping, Sports Wear, Accessories
Synagogues, Religious Organizations
Pubs, Restaurants, Italian, Bars, American (Traditional), Nightlife, Greek
Ice Cream & Frozen Yogurt, Fast Food, Burgers, Restaurants, Food


In [12]:
# ============================================
# STEP 4.6 — CATEGORY FREQUENCY ANALYSIS
# ============================================

all_categories = (
    business_sample["categories"]
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
)

print("Total category assignments:", len(all_categories))
print("Unique categories:", all_categories.nunique())

print("\nTop 20 categories:")
print(all_categories.value_counts().head(20))

Total category assignments: 22374
Unique categories: 944

Top 20 categories:
categories
Restaurants                  1762
Food                          917
Shopping                      841
Beauty & Spas                 485
Home Services                 466
Nightlife                     412
Local Services                382
Bars                          373
Health & Medical              372
Automotive                    350
Event Planning & Services     326
Active Life                   286
Sandwiches                    279
American (Traditional)        259
Coffee & Tea                  251
Pizza                         250
Fast Food                     218
Breakfast & Brunch            218
Fashion                       210
American (New)                208
Name: count, dtype: int64


## 5. Load the Complete Business Dataset

In [13]:
# ============================================
# STEP 5.1 — LOAD COMPLETE BUSINESS DATA SAFELY
# ============================================

import json
import pandas as pd

business_rows = []

with open(business_file, "r", encoding="utf-8") as file:

    for line in file:
        business_rows.append(json.loads(line))

business_df = pd.DataFrame(business_rows)

print("Complete business dataset loaded successfully.")
print("Rows:", len(business_df))
print("Columns:", len(business_df.columns))
print(
    "Memory usage:",
    f"{business_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Complete business dataset loaded successfully.
Rows: 150346
Columns: 14
Memory usage: 149.41 MB


## 6. Data Quality Assessment

In [14]:
# ============================================
# STEP 5.2 — DUPLICATE BUSINESS ID CHECK
# ============================================

duplicate_business_ids = business_df["business_id"].duplicated().sum()

print("Duplicate business IDs:", duplicate_business_ids)

Duplicate business IDs: 0


In [15]:
# ============================================
# STEP 5.3 — MISSING VALUE CHECK
# ============================================

missing_values = business_df.isnull().sum()

missing_summary = (
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
)

print("Missing values by column:")
print(missing_summary)

Missing values by column:
hours         23223
attributes    13744
categories      103
dtype: int64


In [16]:
# ============================================
# STEP 5.4 — INSPECT MISSING CATEGORIES
# ============================================

missing_category_businesses = business_df[
    business_df["categories"].isna()
]

print("Businesses with missing categories:",
      len(missing_category_businesses))

print("\nSample records:")
display(
    missing_category_businesses[
        ["business_id", "name", "city", "state", "stars", "review_count", "is_open", "categories"]
    ].head(10)
)

Businesses with missing categories: 103

Sample records:


,business_id,name,city,state,stars,review_count,is_open,categories
1917,SMYXOLPyM95JvZ-oqnsWUA,A A Berlin Glass & Mirror Co,Berlin,NJ,3.0,5,1,NaN
2243,9ryVeDaaR-le3kiSayTGow,Pauline African Hair Braiding & Weaving,Saint Ann,MO,1.0,5,1,NaN
3304,xT3J-SP5g49g2FjQfLEQfg,Luxury Perfume,Reno,NV,2.0,5,1,NaN
3324,_obl2-rphXvtzP3y_ekV1Q,Certegy Payment Services,Saint Petersburg,FL,1.0,7,1,NaN
4640,mKxCNYEoKt6d_1rXmvRwww,Green Envy,Saint Charles,MO,1.5,5,1,NaN
5478,9QoKKDZB_YuDeS5TxRW8bg,Our 365 Portraits,Saint Louis,MO,1.0,10,1,NaN
10042,lxaSo0sBK36BNDRL6uWHXg,Parklane Management Company,Boise,ID,1.0,5,1,NaN
12526,ZERQMWb1PFzCfbfknqq-fA,Pilot Air Freight,Media,PA,1.5,8,1,NaN
13023,cs7i8-NtrT2P4dMYa2fX-g,Direct USA,Tampa,FL,1.0,5,1,NaN
13165,tfQEd3kakCQdbjfdp62rzg,Nicholson's College Cars,Marrero,LA,2.5,5,1,NaN


In [17]:
# ============================================
# STEP 5.5 — HANDLE MISSING CATEGORIES
# ============================================

before_count = len(business_df)

business_df = business_df.dropna(subset=["categories"]).copy()

after_count = len(business_df)

print("Rows before cleaning:", before_count)
print("Rows removed:", before_count - after_count)
print("Rows after cleaning:", after_count)
print("Missing categories remaining:", business_df["categories"].isna().sum())

Rows before cleaning: 150346
Rows removed: 103
Rows after cleaning: 150243
Missing categories remaining: 0


In [18]:
print(business_df.dtypes)

business_id         str
name                str
address             str
city                str
state               str
postal_code         str
latitude        float64
longitude       float64
stars           float64
review_count      int64
is_open           int64
attributes       object
categories          str
hours            object
dtype: object


In [19]:
# ============================================
# STEP 5.7 — VALIDATE CORE NUMERICAL FIELDS
# ============================================

print("Stars outside 1–5:",
      ((business_df["stars"] < 1) | (business_df["stars"] > 5)).sum())

print("Negative review counts:",
      (business_df["review_count"] < 0).sum())

print("Invalid is_open values:",
      (~business_df["is_open"].isin([0, 1])).sum())

print("Missing business IDs:",
      business_df["business_id"].isna().sum())

print("Missing business names:",
      business_df["name"].isna().sum())

Stars outside 1–5: 0
Negative review counts: 0
Invalid is_open values: 0
Missing business IDs: 0
Missing business names: 0


In [20]:
# ============================================
# STEP 5.8 — TEXT FORMATTING CHECK
# ============================================

print("City values with leading/trailing spaces:",
      (business_df["city"] != business_df["city"].str.strip()).sum())

print("State values with leading/trailing spaces:",
      (business_df["state"] != business_df["state"].str.strip()).sum())

print("Business names with leading/trailing spaces:",
      (business_df["name"] != business_df["name"].str.strip()).sum())

City values with leading/trailing spaces: 49
State values with leading/trailing spaces: 0
Business names with leading/trailing spaces: 150


In [21]:
# ============================================
# STEP 5.9 — CLEAN TEXT WHITESPACE
# ============================================

business_df["city"] = business_df["city"].str.strip()
business_df["state"] = business_df["state"].str.strip()
business_df["name"] = business_df["name"].str.strip()

print("Whitespace cleaning completed.")

Whitespace cleaning completed.


## 8. Category Normalization and Data Modelling

The comma-separated category field is normalized into a category dimension and a business-category relationship table. This structure supports relational analysis in SQL Server and Power BI.

In [22]:
# ============================================
# STEP 6.1 — INSPECT CATEGORY FORMAT
# ============================================

print(business_df["categories"].head(10).to_string(index=False))

Doctors, Traditional Chinese Medicine, Naturopa...
Shipping Centers, Local Services, Notaries, Mai...
Department Stores, Shopping, Fashion, Home & Ga...
Restaurants, Food, Bubble Tea, Coffee & Tea, Ba...
                         Brewpubs, Breweries, Food
Burgers, Fast Food, Sandwiches, Food, Ice Cream...
Sporting Goods, Fashion, Shoe Stores, Shopping,...
               Synagogues, Religious Organizations
Pubs, Restaurants, Italian, Bars, American (Tra...
Ice Cream & Frozen Yogurt, Fast Food, Burgers, ...


In [23]:
# ============================================
# STEP 6.1 — NORMALIZE CATEGORIES
# ============================================

business_category_df = (
    business_df[["business_id", "categories"]]
    .assign(category=lambda x: x["categories"].str.split(","))
    .explode("category")
)

business_category_df["category"] = (
    business_category_df["category"]
    .str.strip()
)

business_category_df = (
    business_category_df[["business_id", "category"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Business-category rows:", len(business_category_df))
print("Unique businesses:", business_category_df["business_id"].nunique())
print("Unique categories:", business_category_df["category"].nunique())

display(business_category_df.head(10))

Business-category rows: 668549
Unique businesses: 150243
Unique categories: 1311


,business_id,category
0,Pns2l4eNsfO8kk83dixA6A,Doctors
1,Pns2l4eNsfO8kk83dixA6A,Traditional Chinese Medicine
2,Pns2l4eNsfO8kk83dixA6A,Naturopathic/Holistic
3,Pns2l4eNsfO8kk83dixA6A,Acupuncture
4,Pns2l4eNsfO8kk83dixA6A,Health & Medical
5,Pns2l4eNsfO8kk83dixA6A,Nutritionists
6,mpf3x-BjTdTEA3yCZrAYPw,Shipping Centers
7,mpf3x-BjTdTEA3yCZrAYPw,Local Services
8,mpf3x-BjTdTEA3yCZrAYPw,Notaries
9,mpf3x-BjTdTEA3yCZrAYPw,Mailbox Centers


### 8.1 Create the Category Dimension

In [24]:
# ============================================
# STEP 6.2 — CREATE CATEGORY TABLE
# ============================================

category_df = (
    business_category_df[["category"]]
    .drop_duplicates()
    .sort_values("category")
    .reset_index(drop=True)
)

category_df["category_id"] = (
    category_df.index + 1
)

category_df = category_df[
    ["category_id", "category"]
]

print("Unique categories:", len(category_df))

display(category_df.head(20))

Unique categories: 1311


,category_id,category
0,1,& Probates
1,2,3D Printing
2,3,ATV Rentals/Tours
3,4,Acai Bowls
4,5,Accessories
5,6,Accountants
6,7,Acne Treatment
7,8,Active Life
8,9,Acupuncture
9,10,Addiction Medicine


In [25]:
# ============================================
# STEP 6.3 — CREATE BUSINESS-CATEGORY RELATIONSHIP
# ============================================

business_category_df = business_category_df.merge(
    category_df,
    on="category",
    how="left"
)

business_category_df = business_category_df[
    ["business_id", "category_id"]
]

print("Business-category relationships:",
      len(business_category_df))

print("Missing category IDs:",
      business_category_df["category_id"].isna().sum())

display(business_category_df.head(10))

Business-category relationships: 668549
Missing category IDs: 0


,business_id,category_id
0,Pns2l4eNsfO8kk83dixA6A,360
1,Pns2l4eNsfO8kk83dixA6A,1201
2,Pns2l4eNsfO8kk83dixA6A,799
3,Pns2l4eNsfO8kk83dixA6A,9
4,Pns2l4eNsfO8kk83dixA6A,552
5,Pns2l4eNsfO8kk83dixA6A,814
6,mpf3x-BjTdTEA3yCZrAYPw,1066
7,mpf3x-BjTdTEA3yCZrAYPw,713
8,mpf3x-BjTdTEA3yCZrAYPw,809
9,mpf3x-BjTdTEA3yCZrAYPw,722


In [26]:
# ============================================
# STEP 6.4 — CREATE FINAL BUSINESS TABLE
# ============================================

business_df["business_status"] = business_df["is_open"].map({
    1: "Open",
    0: "Closed"
})

business_final_df = business_df[
    [
        "business_id",
        "name",
        "address",
        "city",
        "state",
        "postal_code",
        "latitude",
        "longitude",
        "stars",
        "review_count",
        "is_open",
        "business_status"
    ]
].copy()

print("Final business rows:", len(business_final_df))
print("Final business columns:", len(business_final_df.columns))

display(business_final_df.head())

Final business rows: 150243
Final business columns: 12


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,business_status
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,Closed
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,Open
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,Closed
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,Open
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,Open


## 10. Export Processed Data

The structured datasets are exported locally for SQL Server ingestion.

**Note:** These processed CSV files should remain outside the public GitHub repository.

In [27]:
# ============================================
# STEP 6.5 — EXPORT PROCESSED DATA
# ============================================

import os

processed_path = "data/processed"

os.makedirs(processed_path, exist_ok=True)

business_final_df.to_csv(
    f"{processed_path}/business.csv",
    index=False
)

category_df.to_csv(
    f"{processed_path}/category.csv",
    index=False
)

business_category_df.to_csv(
    f"{processed_path}/business_category.csv",
    index=False
)

print("Processed data exported successfully.")
print("\nFiles created:")

for file in os.listdir(processed_path):
    print("-", file)

Processed data exported successfully.

Files created:
- business.csv
- business_category.csv
- category.csv


## 11. Python → SQL Server Integration

The cleaned analytical tables are loaded into Microsoft SQL Server using **SQLAlchemy and PyODBC**.

The connection below uses Windows Authentication and the local SQL Server instance used during project development.

In [28]:
# Install the Python SQL Server connection packages
# SQLAlchemy → Python database interface
# pyodbc     → SQL Server driver connection

In [41]:
# ============================================
# STEP 7.2 — TEST SQL SERVER CONNECTION
# ============================================

from sqlalchemy import create_engine, text

SERVER = r"LAPTOP-O36NJ3AR\SQLEXPRESS"
DATABASE = "Yelp_Business_Analytics"
DRIVER = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

with engine.connect() as connection:
    result = connection.execute(text("SELECT @@VERSION"))
    print("SQL Server connection successful!")
    print(result.fetchone()[0])

SQL Server connection successful!
Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Home Single Language 10.0 <X64> (Build 26200: ) (Hypervisor)



In [30]:
# ============================================
# STEP 7.3 — CREATE PROJECT DATABASE
# ============================================

from sqlalchemy import text

with engine.connect() as connection:
    connection.execution_options(isolation_level="AUTOCOMMIT").execute(
        text("""
        IF DB_ID('Yelp_Business_Analytics') IS NULL
        BEGIN
            CREATE DATABASE Yelp_Business_Analytics
        END
        """)
    )

print("Database 'Yelp_Business_Analytics' is ready.")

Database 'Yelp_Business_Analytics' is ready.


In [31]:
# ============================================
# STEP 7.4 — CONNECT TO PROJECT DATABASE
# ============================================

database = "Yelp_Business_Analytics"

connection_string_db = (
    "mssql+pyodbc://@"
    + server
    + "/"
    + database
    + "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine_db = create_engine(connection_string_db)

with engine_db.connect() as connection:
    result = connection.execute(text("SELECT DB_NAME()"))
    print("Connected database:", result.fetchone()[0])

Connected database: Yelp_Business_Analytics


### 11.1 Load Business Table

In [32]:
# ============================================
# STEP 7.5 — CREATE BUSINESS TABLE
# ============================================

business_final_df.head(0).to_sql(
    "business",
    engine_db,
    if_exists="replace",
    index=False,
    dtype={
        "business_id": __import__("sqlalchemy").types.VARCHAR(30),
        "name": __import__("sqlalchemy").types.NVARCHAR(255),
        "address": __import__("sqlalchemy").types.NVARCHAR(255),
        "city": __import__("sqlalchemy").types.NVARCHAR(100),
        "state": __import__("sqlalchemy").types.VARCHAR(10),
        "postal_code": __import__("sqlalchemy").types.VARCHAR(20),
        "latitude": __import__("sqlalchemy").types.Float,
        "longitude": __import__("sqlalchemy").types.Float,
        "stars": __import__("sqlalchemy").types.Float,
        "review_count": __import__("sqlalchemy").types.Integer,
        "is_open": __import__("sqlalchemy").types.Integer,
        "business_status": __import__("sqlalchemy").types.VARCHAR(10)
    }
)

print("Business table created successfully.")

Business table created successfully.


In [33]:
# ============================================
# STEP 7.6 — LOAD BUSINESS DATA
# ============================================

business_final_df.to_sql(
    "business",
    engine_db,
    if_exists="append",
    index=False,
    chunksize=5000
)

print("Business data loaded successfully.")
print("Rows loaded:", len(business_final_df))

Business data loaded successfully.
Rows loaded: 150243


In [34]:
# ============================================
# STEP 7.7 — VALIDATE BUSINESS TABLE
# ============================================

with engine_db.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM business")
    )

    sql_business_count = result.scalar()

print("Python rows:", len(business_final_df))
print("SQL Server rows:", sql_business_count)

print(
    "Row count validation:",
    "PASSED" if len(business_final_df) == sql_business_count else "FAILED"
)

Python rows: 150243
SQL Server rows: 150243
Row count validation: PASSED


### 11.2 Load Category Table

In [35]:
# ============================================
# STEP 7.8 — CREATE CATEGORY TABLE
# ============================================

from sqlalchemy import types

category_df.head(0).to_sql(
    "category",
    engine_db,
    if_exists="replace",
    index=False,
    dtype={
        "category_id": types.Integer,
        "category": types.NVARCHAR(150)
    }
)

print("Category table created successfully.")

Category table created successfully.


In [36]:
# ============================================
# STEP 7.9 — LOAD CATEGORY DATA
# ============================================

category_df.to_sql(
    "category",
    engine_db,
    if_exists="append",
    index=False
)

print("Category data loaded successfully.")
print("Rows loaded:", len(category_df))

Category data loaded successfully.
Rows loaded: 1311


In [37]:
# ============================================
# STEP 7.10 — VALIDATE CATEGORY TABLE
# ============================================

with engine_db.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM category")
    )

    sql_category_count = result.scalar()

print("Python categories:", len(category_df))
print("SQL Server categories:", sql_category_count)

print(
    "Row count validation:",
    "PASSED" if len(category_df) == sql_category_count else "FAILED"
)

Python categories: 1311
SQL Server categories: 1311
Row count validation: PASSED


### 11.3 Load Business-Category Relationship

In [38]:
# ============================================
# STEP 7.11 — CREATE BUSINESS-CATEGORY TABLE
# ============================================

business_category_df.head(0).to_sql(
    "business_category",
    engine_db,
    if_exists="replace",
    index=False,
    dtype={
        "business_id": types.VARCHAR(30),
        "category_id": types.Integer
    }
)

print("Business-category table created successfully.")

Business-category table created successfully.


In [39]:
# ============================================
# STEP 7.12 — LOAD BUSINESS-CATEGORY DATA
# ============================================

business_category_df.to_sql(
    "business_category",
    engine_db,
    if_exists="append",
    index=False,
    chunksize=10000
)

print("Business-category data loaded successfully.")
print("Rows loaded:", len(business_category_df))

Business-category data loaded successfully.
Rows loaded: 668549


## 12. Final Python → SQL Server Validation

The final validation confirms that SQL Server row counts match the datasets generated in Python.

In [40]:
# ============================================
# STEP 7.13 — FINAL SQL DATA VALIDATION
# ============================================

validation_query = text("""
SELECT 'business' AS table_name, COUNT(*) AS row_count
FROM business

UNION ALL

SELECT 'category', COUNT(*)
FROM category

UNION ALL

SELECT 'business_category', COUNT(*)
FROM business_category
""")

with engine_db.connect() as connection:
    validation_result = connection.execute(validation_query)

    for row in validation_result:
        print(f"{row.table_name}: {row.row_count:,} rows")

business: 150,243 rows
category: 1,311 rows
business_category: 668,549 rows


## 13. Python Stage Complete

The Python workflow has completed data acquisition setup, profiling, cleaning, category normalization, validation, export, SQL Server connectivity, loading, and cross-platform validation.

The validated SQL Server database is then used for SQL business analysis and Power BI reporting.